In [ ]:
import os
import sys
import h5py
import hdf5plugin
import numpy as np
from numpy.fft import fft2, ifft2, fftshift
import torch
from scipy.ndimage import fourier_shift

import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import cmcrameri.cm as cmc

import ptychi.image_proc as ip

%matplotlib inline

### Detector geometry

In [ ]:
def detector_q_geometry(
    detector_shape,
    pixel_size_m,
    energy_kev,
    detector_distance_m,
    beam_center_px=None,
):
    """Calculate reciprocal-space coordinates for a square-pixel detector.

    Assumes transmission geometry with a flat detector perpendicular to the
    incident beam. ``beam_center_px`` is given as (row, column); when omitted,
    the geometric center of the detector is used. Q values are in inverse
    angstroms.
    """
    detector_shape = tuple(detector_shape)
    if len(detector_shape) != 2 or any(n <= 0 for n in detector_shape):
        raise ValueError("detector_shape must be a pair of positive integers")
    if pixel_size_m <= 0 or energy_kev <= 0 or detector_distance_m <= 0:
        raise ValueError("pixel size, energy, and detector distance must be positive")

    ny, nx = detector_shape
    if beam_center_px is None:
        beam_center_px = ((ny - 1) / 2, (nx - 1) / 2)
    center_row, center_col = map(float, beam_center_px)

    wavelength_angstrom = 12.398419843320026 / energy_kev
    wave_number = 2 * np.pi / wavelength_angstrom
    rows, cols = np.indices((ny, nx), dtype=float)
    x_m = (cols - center_col) * pixel_size_m
    y_m = (rows - center_row) * pixel_size_m
    ray_length_m = np.sqrt(x_m**2 + y_m**2 + detector_distance_m**2)

    # q = k_out - k_in, with the incident beam along +z.
    qx = wave_number * x_m / ray_length_m
    qy = wave_number * y_m / ray_length_m
    qz = wave_number * (detector_distance_m / ray_length_m - 1)
    q = np.sqrt(qx**2 + qy**2 + qz**2)

    two_theta_one_pixel = np.arctan2(pixel_size_m, detector_distance_m)
    q_per_pixel_center = 2 * wave_number * np.sin(two_theta_one_pixel / 2)
    q_per_pixel_small_angle = wave_number * pixel_size_m / detector_distance_m

    return {
        "wavelength_angstrom": wavelength_angstrom,
        "q_Ainv": q,
        "qx_Ainv": qx,
        "qy_Ainv": qy,
        "qz_Ainv": qz,
        "q_range_Ainv": (float(q.min()), float(q.max())),
        "q_per_pixel_center_Ainv": float(q_per_pixel_center),
        "q_per_pixel_small_angle_Ainv": float(q_per_pixel_small_angle),
    }


q_geometry = detector_q_geometry(
    detector_shape=(256, 256),
    pixel_size_m=150e-6,
    energy_kev=9.659,
    detector_distance_m=2.06,
    beam_center_px=(127.5, 127.5),
)
print("Q range [A^-1]:", q_geometry["q_range_Ainv"])
print("Q/pixel near center [A^-1]:", q_geometry["q_per_pixel_center_Ainv"])

In [ ]:
def plot_detector_q(
    detector_image,
    q_geometry,
    *,
    ax=None,
    cmap=cmc.lapaz,
    norm=None,
    colorbar=True,
    colorbar_label="Intensity",
    detector_rows_down=True,
):
    """Plot a detector image using the calculated Qx and Qy coordinates."""
    detector_image = np.asarray(detector_image)
    qx = np.asarray(q_geometry["qx_Ainv"])
    qy = np.asarray(q_geometry["qy_Ainv"])

    if detector_image.ndim != 2:
        raise ValueError("detector_image must be a 2D array")
    if qx.shape != detector_image.shape or qy.shape != detector_image.shape:
        raise ValueError("detector_image and Q-coordinate maps must have the same shape")

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5), dpi=300, constrained_layout=True)
    else:
        fig = ax.figure

    image_artist = ax.pcolormesh(
        qx,
        qy,
        detector_image,
        shading="nearest",
        cmap=cmap,
        norm=norm,
        rasterized=True,
    )
    ax.set_xlabel(r"$Q_x$ ($\mathrm{\AA}^{-1}$)")
    ax.set_ylabel(r"$Q_y$ ($\mathrm{\AA}^{-1}$)")
    ax.set_aspect("equal")

    # Preserve the usual detector-image convention: row zero is at the top.
    if detector_rows_down:
        ax.invert_yaxis()

    q_min, q_max = q_geometry["q_range_Ainv"]
    ax.set_title(rf"Detector frame ($|Q|={q_min:.3g}$ to ${q_max:.3g}\ \mathrm{{\AA}}^{{-1}}$)")

    if colorbar:
        cbar = fig.colorbar(image_artist, ax=ax)
        cbar.set_label(colorbar_label)

    return fig, ax, image_artist

## Zone-plate stage displacement per binned detector pixel

This scale is the physical zone-plate-stage displacement corresponding to **one pixel of motion in the detector image**.

For X-ray energy $E$ in keV, the wavelength is

$$
\lambda\,[\mathrm{m}] = \frac{1.239841984\times10^{-9}}{E\,[\mathrm{keV}]}.
$$

For a first-order zone plate with diameter $D$ and outermost-zone width $\Delta r$, the paraxial focal length is

$$
f \simeq \frac{D\,\Delta r}{\lambda}.
$$

Let $\delta$ be the sample defocus, positive when the sample is downstream of focus, and let $L$ be the sample-to-detector distance. The zone-plate-to-detector distance is

$$
z = f + \delta + L.
$$

A lateral zone-plate displacement $\Delta X_{\mathrm{ZP}}$ changes the outgoing angle by $\Delta\theta=\Delta X_{\mathrm{ZP}}/f$. A feature fixed in the upstream illumination therefore moves on the detector by

$$
\Delta x_{\mathrm{det}}
= z\,\Delta\theta
= \frac{z}{f}\,\Delta X_{\mathrm{ZP}}.
$$

The detector displacement gain is consequently

$$
G = \frac{\Delta x_{\mathrm{det}}}{\Delta X_{\mathrm{ZP}}}
  = \frac{f+\delta+L}{f}.
$$

For raw detector pixel size $p_{\mathrm{det}}$ and detector binning factor $b$, the effective detector pitch is $p_{\mathrm{det,eff}}=b\,p_{\mathrm{det}}$. The desired stage displacement per binned detector pixel is

$$
\boxed{
s_{\mathrm{eff}}
= \frac{p_{\mathrm{det,eff}}}{G}
= \frac{b\,p_{\mathrm{det}}\,f}{f+\delta+L}
}.
$$

Physical stage positions are converted to detector-pixel shifts using

$$
\Delta n_{\mathrm{det}}
= \frac{\Delta X_{\mathrm{ZP}}}{s_{\mathrm{eff}}}.
$$

To register a moving background, apply the negative of this displacement, subject to the detector/stage axis ordering and sign conventions.


In [ ]:
def zone_plate_stage_pixel_scale(
    energy_kev,
    zone_plate_diameter_m,
    outer_zone_width_m,
    sample_detector_distance_m,
    defocus_m,
    detector_pixel_size_m,
    bin_factor=1,
):
    """Calculate zone-plate-stage metres per binned detector pixel.

    Assumes first-order zone-plate focusing, paraxial propagation, and that
    defocus_m is positive when the sample is downstream of the focal plane.
    """
    positive_inputs = {
        "energy_kev": energy_kev,
        "zone_plate_diameter_m": zone_plate_diameter_m,
        "outer_zone_width_m": outer_zone_width_m,
        "sample_detector_distance_m": sample_detector_distance_m,
        "detector_pixel_size_m": detector_pixel_size_m,
    }
    for name, value in positive_inputs.items():
        if not np.isfinite(value) or value <= 0:
            raise ValueError(f"{name} must be finite and positive; got {value}")
    if not isinstance(bin_factor, (int, np.integer)) or bin_factor < 1:
        raise ValueError("bin_factor must be a positive integer")
    if not np.isfinite(defocus_m):
        raise ValueError("defocus_m must be finite")

    wavelength_m = 1.239841984e-9 / energy_kev
    focal_length_m = (
        zone_plate_diameter_m * outer_zone_width_m / wavelength_m
    )
    zone_plate_detector_distance_m = (
        focal_length_m + defocus_m + sample_detector_distance_m
    )
    if zone_plate_detector_distance_m <= 0:
        raise ValueError("The calculated zone-plate-to-detector distance must be positive")

    detector_shift_gain = zone_plate_detector_distance_m / focal_length_m
    binned_detector_pixel_size_m = detector_pixel_size_m * bin_factor
    stage_displacement_per_detector_pixel_m = (
        binned_detector_pixel_size_m / detector_shift_gain
    )

    return {
        "wavelength_m": float(wavelength_m),
        "focal_length_m": float(focal_length_m),
        "zone_plate_detector_distance_m": float(zone_plate_detector_distance_m),
        "detector_shift_gain": float(detector_shift_gain),
        "binned_detector_pixel_size_m": float(binned_detector_pixel_size_m),
        "stage_displacement_per_detector_pixel_m": float(
            stage_displacement_per_detector_pixel_m
        ),
    }


zp_stage_geometry = zone_plate_stage_pixel_scale(
    energy_kev=9.659,
    zone_plate_diameter_m=200e-6,
    outer_zone_width_m=20e-9,
    sample_detector_distance_m=2.06,
    defocus_m=100e-6,
    detector_pixel_size_m=75e-6,
    bin_factor=2,
)
stage_m_per_detector_px = zp_stage_geometry[
    "stage_displacement_per_detector_pixel_m"
]

positions = np.genfromtxt("/home/aileenluo/ptycho-vit/workspace/pos_14pix_40000.csv", delimiter=",")
print(positions.shape)
positions = positions.T
positions_detector_px = positions / stage_m_per_detector_px
alignment_shifts_px = -positions_detector_px

print("Focal length [m]:", zp_stage_geometry["focal_length_m"])
print("Detector shift gain:", zp_stage_geometry["detector_shift_gain"])
print("Stage displacement per binned detector pixel [m]:", stage_m_per_detector_px)

### Load data

In [ ]:
# Test 273 reconstruction preprocess
with h5py.File("/mnt/localdisk/scratch/aileenluo/raw/S26_20260206/scan_273_200271.h5", "r") as f:
    data = f["/entry/data/data"][::20, 550:1062, 258:770][()]
print(data.dtype)
data = data.astype(np.int32)
data[data < 0] = 0
data = data.reshape(-1, 256, 2, 256, 2).sum(axis=(2, 4))
data = np.sqrt(data.astype(np.float32))

In [ ]:
positions = np.genfromtxt("/home/aileenluo/ptycho-vit/workspace/pos_14pix_40000.csv", delimiter=",")
print(positions.shape)
positions = positions.T

## Weakly scattering samples

In [ ]:
diff_amp = data
positions_detector_px = positions[:, ::20] / stage_m_per_detector_px
print(positions_detector_px[0].max(), positions_detector_px[1].max())
diff_mean = np.mean(diff_amp, 0)
diff_shifted = np.zeros_like(diff_amp)
diff_mean_shifted = np.zeros_like(diff_amp)
for i in range(diff_amp.shape[0]):
    diff_shifted[i] = np.abs(ifft2(fourier_shift(fft2(diff_amp[i]), [-positions_detector_px[0, i], -positions_detector_px[1, i]])))
    diff_mean_shifted[i] = np.abs(ifft2(fourier_shift(fft2(diff_mean), [-positions_detector_px[0, i], -positions_detector_px[1, i]])))
flatfield = np.sum(diff_shifted, 0) / np.sum(diff_mean_shifted, 0)
diff_corrected = np.zeros_like(diff_amp)
for i in range(diff_amp.shape[0]):
    diff_corrected[i] = diff_amp[i] / np.abs(ifft2(fourier_shift(fft2(flatfield), [positions_detector_px[0, i], positions_detector_px[1, i]])))

In [ ]:
f, ax = plt.subplots(ncols=3, dpi=300)
ax[0].imshow(diff_amp[12], interpolation='none')
ax[1].imshow(diff_corrected[12], interpolation='none')
ax[2].imshow(diff_corrected[12]-diff_amp[12], interpolation='none')

In [ ]:
f, ax, image_artist = plot_detector_q(
    flatfield,
    q_geometry
)
#f.savefig("flatfield.svg", transparent=True, bbox_inches="tight", dpi=300)

## Strongly scattering (still in testing)

In [ ]:
positions_pixels = np.vstack((probe_position_y_m, probe_position_x_m)).T / (6.685e-9)
print(positions_pixels.shape, positions_pixels)
object_patches = ip.extract_patches_fourier_shift(
    torch.from_numpy(obj[0]), torch.from_numpy(positions_pixels), (256, 256)
)
object_patches.shape

In [ ]:
diff_amp = np.sqrt(data)
print(diff_amp.dtype)
positions = np.vstack((probe_position_y_m, probe_position_x_m)) / 0.78e-6
diff_reconstructed = np.zeros_like(diff_amp)
print(diff_reconstructed.shape)
for i in range(diff_reconstructed.shape[0]):
    diff_reconstructed[i] = np.sqrt((np.abs(fftshift(fft2(object_patches[i].numpy() * probe)))**2).sum(1)[0])

diff_shifted = np.zeros_like(diff_amp)
diff_recon_shifted = np.zeros_like(diff_amp)
for i in range(diff_amp.shape[0]):
    diff_shifted[i] = np.abs(ifft2(fourier_shift(fft2(diff_amp[i]), [-positions[0, i], -positions[1, i]])))
    diff_recon_shifted[i] = np.abs(ifft2(fourier_shift(fft2(diff_reconstructed[i]), [-positions[0, i], -positions[1, i]])))
flatfield_strong = np.sum(diff_shifted, 0) / np.sum(diff_recon_shifted, 0)
diff_corrected_strong = np.zeros_like(diff_amp)
for i in range(diff_amp.shape[0]):
    diff_corrected_strong[i] = diff_amp[i] / np.abs(ifft2(fourier_shift(fft2(flatfield_strong), [positions[0, i], positions[1, i]])))